<a id="top"></a>
## The Rainbow Connection: Spectral Classification and Multi-Mission Searches with SDSS
<img src="figures/rainbow_banner.png" alt="alt text here" width="100%">

MAST Summer Webinar Series: 2026 July 21, 11:00-12:00 EDT
***

## Learning Goals

By the end of this webinar, you will:

- Learn how to use `astroquery.mast` to search Multi-Mission data from the MAST archive
- Understand how to download and plot a stellar spectrum and label its absorption lines
- Learn about spectral classification and how to classify stars

## Table of Contents
* [Introduction](#Introduction)
  * [Imports](#Imports)
* [The Rainbow Connection: Searching MAST with astroquery](#The-Rainbow-Connection:-Searching-MAST-with-astroquery
)
  * [Searching MAST with astroquery](#Searching-MAST-with-astroquery)
  * [Downloading the Data Products](#Downloading-the-Data-Products)
  * [**Exercise #1:** Building a query (SDSS)](#Exercise-1:-Building-a-query)
* [Unweaving a Rainbow: Plotting Stellar Spectra](#Unweaving-a-Rainbow:-Plotting-Stellar-Spectra)
  * [About SDSS](#About-SDSS)
  * [Plotting a Spectrum](#Plotting-a-Spectrum)
  * [Spectral Classification](#Spectral-Classification)
  * [Exercise #2: Plot the HST Spectrum](#Exercise-2:-Plot-the-HST-Spectrum)
  * [Enhancing the Plot](#Enhancing-the-Plot)

* [The Learn'd Astronomer: Spectral Classification](#The-Learn’d-Astronomer:-Spectral-Classification)
  * [Classifying a Star](#Classifying-a-Star)
  * [Exercise #3:** Classify a random star](#Exercise-3:-Classify-a-random-star)

* [Conclusions: Responsibility to Awe](#Conclusions-Responsibility-to-Awe)
  * [Exercise Solutions](#Exercise-Solutions)
  * [Additional Resources](#Additional-Resources)
  * [About this Notebook](#About-this-Notebook)


## Introduction

The [Mikulski Archive for Space Telescopes (MAST)](https://archive.stsci.edu) is an astronomical data archive that contains data from over 20 different missions, including NASA’s flagship space telescopes like HST and JWST. As of June 2026, MAST contains over 400 million astronomical observations - which includes images, spectra, light curves, and catalogs for astronomical research! Here are some of the telesopes which have data at MAST: how many do you recognize?

<img src="figures/mast_missions.png" alt="alt text here" width="100%">

This webinar session will focus on *spectra*: measuring the flux of a target as a function of wavelength. The spectrum of a star or galaxy can tell you a lot about it: it's temperature, chemical composition, velocities, and even its age or evolution history.

By the end of this session, you will learn how to search MAST's collections for spectra, and use them to classify stars!



### Imports
These are the packages we will need for this tutorial. Run this cell to import them!

- *numpy* to handle array functions and some basic math
- *matplotlib* for plotting data and creating figures
- *astropy.io* for reading FITS files
- *astroquery.mast* to search and download data from MAST


In [ ]:
%matplotlib inline

import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from astropy.io import fits
from astropy.table import Table

from astroquery.mast import Observations

This cell updates some of the settings in `matplotlib` to use larger font sizes in the figures:

In [ ]:
# Update Plotting Parameters
params = {
    "axes.labelsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "text.usetex": False,
    "lines.linewidth": 1,
    "axes.titlesize": 18,
    "font.family": "serif",
    "font.size": 12,
}
plt.rcParams.update(params)

***

## The Rainbow Connection: Searching MAST with astroquery

> *Why are there so many*<br>
> *Songs about rainbows*<br>
> *And what's on the other side?*<br>
> *Rainbows are visions*<br>
> *But only illusions*<br>
> *And rainbows have nothing to hide*<br>
>
> *What's so amazing that keeps us stargazing?*<br>
> *And what do we think we might see?*<br>
> *Someday we'll find it*<br>
> *The rainbow connection*<br>
> *The lovers, the dreamers, and me*<br>
> <br>
> -Kermit the Frog, *The Rainbow Connection*
> <br>
> <br>

The easiest way to search for spectra in MAST is to use `astroquery.mast` - a Python package for searching the data collections avaible in MAST! In this section, we will learn how to use astroquery to search for spectra in the MAST archive.

If you want to learn more, [the documentation and user manual for astroquery.mast is available here!](https://astroquery.readthedocs.io/en/latest/mast/mast.html)


### Searching MAST with astroquery

The main function we will use to search for data is called `Observations.query_criteria`. Here is an example query to search for JWST spectra!

Here's an overview of the keywords used in this function:
- `obs_collection` to search by missions name: "JWST"
- `dataproduct_type` to search only for spectra
- `pagesize` and `page` to limit the search to the first 10 results.


In [ ]:
# Query for JWST Spectra Data
spectro_data = Observations.query_criteria(
    obs_collection="JWST", dataproduct_type="spectrum", pagesize=10, page=1
)

# Display first 10 entries
spectro_data


The table above provides some basic information for each observation:
- `collection`: Tells us the mission for this data, which is `JWST`
- `s_ra` and `s_dec`: Celestial coordinates right ascension and declination.
- `instrument_name`: indicates which instrument was used for the observation. For JWST, this is typically "NIRSPEC" or "MIRI". 
- `t_min` and `t_max`: The modified Julian dates indicating the start and end times of the exposures.
- `wavelength_region`: Indicates the region of the electromagnetic spectrum observed. In this case, `INFRARED`, since JWST observed in the infrared wavelength range.
- `em_min` and `em_max`: The minimum and maximum wavelengths in the spectrum.

Next, let's try a more targeted search. I want to find a spectrum of the star "Tau Ceti". I don't know the coordinates of this star, but I can find them using the `Observations.resolve_object()` function.

In [ ]:
coordinates = Observations.resolve_object("Tau Ceti")

print(coordinates)

Now that we have the coordinates, we can search for HST spectra of that star! The keyword `radius=0` ensures that we only get data at those coordinates (and not another nearby star)

In [ ]:
# Query for HST Spectra Data of Tau Ceti
coordinates = Observations.resolve_object("Tau Ceti")

spectro_data = Observations.query_criteria(
    coordinates=coordinates,
    radius=0,
    obs_collection="HST",
    dataproduct_type="spectrum",
    pagesize=10,
    page=1,
)

# Display first 10 entries
spectro_data


### Downloading the Data Products

If the search results look good, we can see what files are available to download using the `Observations.get_product_list()` funciton and passing it our search results:

In [ ]:
products = Observations.get_product_list(spectro_data)

products

There's a lot of files here, but a lot of them aren't important (calibration files, preview images, etc.) We can download just the spectrum science file using:

In [ ]:
# Downloading spectrum files
manifest = Observations.download_products(
    products,  # Specify a product list to download
    flat=True,  # Download with a "flat" structure in the current directory
    mrp_only=True,  # Limit to Minimum Recommended Products = the primary science file
)

If you've been following along correctly, this would have downloaded two spectra files, named `hst_8143_stis_hd10700_e140m_o5cy_cspec.fits` and `hst_8143_stis_hd10700_e140m_o5cy01_cspec.fits` !

In [ ]:
!ls hst*

### **Exercise 1:** Building a query

Now it's your turn! In the cell below, write a piece of code that will search MAST and download data that meets the following criteria:

- Use astroquery to find results for the target at coordinates (308.81669	76.546953). This star was selected for this tutorial because it is a G2 type main sequence star, very similar to the Sun.
- Search only for data from the "SDSS" mission (Sloan Digital Sky Survey)
- Limit the results to spectra data
- List the products available and download the spectrum file.

In [ ]:
# Write your code here!
coordinates = "308.81669, 76.546953"

# Search MAST for spectra data from SDSS of this star

# Retrieve the available products

# Download the spectrum files


In [ ]:
# EXERCISE 1 SOLUTION
coordinates = "308.81669, 76.546953"

# Search MAST for spectra data from SDSS of this star
spectro_data = Observations.query_criteria(
    coordinates=coordinates,
    radius=0,
    obs_collection="SDSS",
    dataproduct_type="spectrum",
)

# Retrieve the available products
products = Observations.get_product_list(spectro_data)

# Download the spectrum files
manifest = Observations.download_products(
    products,  # Specify a product list to download
    flat=True,  # Download with a "flat" structure in the current directory
    mrp_only=True,  # Limit to Minimum Recommended Products = the primary science file
)

If you did it correctly, you should have downloaded a file named `spec-1660-53230-0023.fits`!

## Unweaving a Rainbow: Plotting Stellar Spectra

> *Philosophy will clip an Angel's wings,* <br>
> *Conquer all mysteries by rule and line,* <br>
> *Empty the haunted air, and gnomed mine—* <br>
> *Unweave a rainbow, as it erewhile made* <br>
> *The tender-person'd Lamia melt into a shade.* <br>
> <br>
> -John Keats, *Lamia*
> <br>
> <br>


### About SDSS
In the first exercise, we downloaded a spectrum from the Sloan Digital Sky Survey (SDSS). SDSS provides an optical-wavelength spectroscopic survey of millions of stars, galaxies, and quasars available through the [SDSS Legacy Archive at MAST](https://archive.stsci.edu/missions-and-data/sdss). This includes data the [Sloan Extension for Galactic Understanding and Exploration (SEGUE)](https://www.sdss4.org/surveys/segue/), which targeted over 400,000 stars in the Milky Way with a goal of mapping out the galactic disk in abundance space. A summary of the SDSS data products available at MAST can be found in the [SDSS Legacy Archive at MAST User Manual](https://outerspace.stsci.edu/display/SDSS/Legacy+Spectra+Data+Products).

### Plotting a Spectrum
Now that we know how to download spectra from the SDSS Legacy Archive at MAST, let's take a closer look at the data. We can print some basic information about the file we just downloaded by opening the data with `astropy.io.fits` and printing the file summary with `.info()`

In [ ]:
# Opening file
spectrum_file = fits.open("spec-1660-53230-0023.fits")

# Display file info
spectrum_file.info()

This file has various extensions, which are explained in the [SDSS Data Model](https://data.sdss.org/datamodel/files/BOSS_SPECTRO_REDUX/RUN2D/spectra/full/PLATE4/spec.html):
- `HDU 0: PRIMARY`: The primary header information. 
- `HDU 1: COADD`: The coadded observed spectrum.
- `HDU 2: SPECOBJ`: Metadata from the SPECOBJ table, which includes targeting information and spectroscopic classifications from the [SDSS Spectra Data Reduction Pipeline](https://www.sdss4.org/dr17/spectro/pipeline/).
- `HDU 3: SPZLINE`: Measurements and line fitting outputs from the [SDSS Spectra Data Analysis Pipeline](https://classic.sdss.org/segue/stellarpars.php), which measures chemical abundances and equivalent widths for absorption lines in the spectrum.
- `HDU 4+`: Individual visit spectra (on red `R1-` and blue `B1-` ccd chips) that were included in the coadded spectrum. 

Now, let's plot the spectrum with `matplotlib.pyplot`. To do this, we'll use the information in the first extension (`COADD`). This extension contains the wavelength and flux information we will need to make a plot.

In [ ]:
# Initiate plot
plt.figure(figsize=(10, 4))

# Load in wavelength and flux
wavelength = 10 ** spectrum_file["COADD"].data["loglam"]
observed_flux = spectrum_file["COADD"].data["FLUX"]
model_flux = spectrum_file["COADD"].data["MODEL"]

# Plot Observed Spectrum
plt.plot(wavelength, observed_flux, c="gray", label="Observed Spectrum")
# Plot Best-fit Model Spectrum
plt.plot(wavelength, model_flux, c="k", label="Model Spectrum")

# Label axes
plt.xlabel(r"Wavelength ($\AA$)")
plt.ylabel("Flux")
plt.title(f"{spectrum_file.filename().strip('./')}")
plt.grid()
plt.legend()


# Set axes limits
plt.xlim(np.min(wavelength), np.max(wavelength))
plt.ylim(0, np.max(observed_flux))

# Show plot
plt.show()

### Spectral Classification

In 1912, astronomer Annie Jump Cannon developed a method to classify stars based on the appearance of their spectrum, which is still used today. Stars are classified as O, B, A, F, G, K, or M based on the strength and location of absorption lines in their spectrum. [[1]](https://articles.adsabs.harvard.edu/pdf/1912AnHar..56..115C) 

SDSS has done all the hard work of classifying the spectra, so we can read this star's classification directly from the file:

In [ ]:
# Print the spectral classification form SDSS
spectrum_file["SPECOBJ"].data["SUBCLASS"][0]


This star is a "G2" star, the same classification as our Sun!

### **Exercise 2:** Plot of the HST Spectrum

Using the HST spectrum we downloaded earlier of Tau-Ceti, let's plot that one next!


In [ ]:
# Open the file
spectrum_file = fits.open("hst_8143_stis_hd10700_e140m_o5cy_cspec.fits")

# Print some info about the file
spectrum_file.info()

# Read in the wavelength and flux values
wavelength = spectrum_file["SCI"].data["WAVELENGTH"][0]
observed_flux = spectrum_file["SCI"].data["FLUX"][0]

# Plot the spectrum using matplotlib here!


In [ ]:
# EXERCISE 2 SOLUTION
# Open the file
spectrum_file = fits.open("hst_8143_stis_hd10700_e140m_o5cy_cspec.fits")

# Print some info about the file
spectrum_file.info()

# Read in the wavelength and flux values
wavelength = spectrum_file["SCI"].data["WAVELENGTH"][0]
observed_flux = spectrum_file["SCI"].data["FLUX"][0]

# Plot the spectrum using matplotlib here!
fig, ax = plt.subplots(figsize=(10, 4))

plt.plot(wavelength, observed_flux, c="gray", label="Observed Spectrum")

plt.grid()

# Label axes
plt.xlabel(r"Wavelength ($\AA$)")
plt.ylabel("Flux")
plt.legend()
plt.title(f"{spectrum_file.filename().strip('./')}")

# Set axes limits
plt.xlim(np.min(wavelength), np.max(wavelength))
plt.ylim(-1e-13, 1e-13)

# Show plot
plt.show()

### Enhancing the Plot

A spectrum is a rainbow, so let's enhance this plot by adding color. Here, we define a function that adds a background to this plot to approximate what this would look like if you could see this spectrum with your eyes.

In [ ]:
def plot_rainbow_background(ax, wavelength, flux):
    """Create a rainbow background on the plot to show the spectrum"""
    # Define a custom rainbow colormap to use for the spectrum
    rainbow_colormap = LinearSegmentedColormap.from_list(
        "",
        ["blueviolet"]
        + [mpl.cm.turbo(x) for x in np.linspace(0.07, 0.999, 10)],
    )

    # Normalize flux from 0 to 1 so all spectra are on uniform scale
    flux = flux / np.nanmax(flux)
    # Define min and max values of the flux to normalize the color scale
    max_flux = np.percentile(flux, 90)
    min_flux = np.percentile(flux, 15)
    # Set the scaling limits of the color map
    # the wavelength of the furthest red and furthest purple
    min_wl = 3800
    max_wl = 6700

    # Define window size (in pixels) to smooth spectrum over
    # (makes absorption lines stand out more)
    window_size = 10
    # Loop through each wavelength
    for wl in wavelength[window_size::]:
        i = np.where(wavelength == wl)[0][0]
        wl_scale = (wl - min_wl) / (max_wl - min_wl)
        flux_at_wl = np.interp(wl, wavelength, flux)
        if i <= window_size:
            flux_at_wl = flux[i]
        else:
            i1 = i - window_size
            i2 = i + window_size + 1
            flux_at_wl = np.nanmin(flux[i1:i2])
        flux_scale = (flux_at_wl - min_flux) / (max_flux - min_flux)

        # Check if flux scale is outside limits
        if flux_scale <= 0:
            flux_scale = 0
        elif flux_scale >= 1:
            flux_scale = 0.999
        # Turn to log scale for more contrast
        elif not np.isfinite(flux_scale):
            flux_scale = np.log10(flux_scale * 9 + 1)

        # Plot as vertical lines
        # Plot in black first for a solid background so lines do not overlap
        ax.axvline(wl, c="k", alpha=1, lw=2, zorder=1)
        # Plot in color
        ax.axvline(
            wl, c=rainbow_colormap(wl_scale), alpha=flux_scale, lw=2, zorder=2
        )

Now, let's use this background-adding function to create a plot:

In [ ]:
# Initiate plot
spectrum_file = fits.open("spec-1660-53230-0023.fits")

fig, ax = plt.subplots(figsize=(10, 3))

wavelength = 10 ** spectrum_file["COADD"].data["loglam"]
observed_flux = spectrum_file["COADD"].data["FLUX"]
model_flux = spectrum_file["COADD"].data["MODEL"]

plt.plot(wavelength, observed_flux, c="w", label="Observed Spectrum", zorder=4)

plt.grid()

# Add classification label
classification = spectrum_file["SPECOBJ"].data["CLASS"][0]
subclass = spectrum_file["SPECOBJ"].data["SUBCLASS"][0]
ax.text(
    0.99,
    0.1,
    f"Classification: {subclass} {classification}",
    fontsize=16,
    ha="right",
    va="top",
    color="w",
    transform=ax.transAxes,
)

plot_rainbow_background(ax, wavelength, model_flux)

# Label axes
plt.xlabel(r"Wavelength ($\AA$)")
plt.ylabel("Flux")
plt.legend(loc="upper right")
plt.title(f"{spectrum_file.filename().strip('./')}")

# Set axes limits
plt.xlim(np.min(wavelength), np.max(wavelength))
plt.ylim(0, np.max(observed_flux))
plt.xscale("log")

# Show plot
plt.show()

It looks pretty! We can see how the background corresponds to the flux values, and the absorption features in the spectrum stick out as vertical black lines in the background. The shortest wavelengths (purple) tend to be brighter than the longest wavelengths (red), but this particular star peaks in green wavelengths, just like our Sun!


## The Learn’d Astronomer: Spectral Classification

>*When I heard the learn’d astronomer,*<br>
>*When the proofs, the figures, were ranged in columns before me,*<br>
>*When I was shown the charts and diagrams, to add, divide, and measure them,*<br>
>*When I sitting heard the astronomer where he lectured with much applause in the lecture-room,*<br>
>*How soon unaccountable I became tired and sick,*<br>
>*Till rising and gliding out I wander’d off by myself,*<br>
>*In the mystical moist night-air, and from time to time,*<br>
>*Look’d up in perfect silence at the stars.*<br>

-Walt Whitman, *When I heard the learn’d astronomer*


### Classifying a Star

Now that we know what the spectrum of a G-type star like the Sun looks like, let's explore the other types of stars!
In this section, we have made a list of stars with different spectral types to plot and explore:



In [ ]:
star_list = [
    {  # O-Star example
        "obs_id": "sdss_spectro_2929-54616-0363",
        "spec_file": "spec-2929-54616-0363.fits",
        "label": "",
    },
    {  # B-star example
        "obs_id": "sdss_spectro_2871-54536-0542",
        "spec_file": "spec-2871-54536-0542.fits",
        "label": "",
    },
    {  # White Dwarf-star example
        "obs_id": "sdss_spectro_2630-54327-0551",
        "spec_file": "spec-2630-54327-0551.fits",
        "label": "White Dwarf Star. ",
    },
    {  # A-star example
        "obs_id": "sdss_spectro_2682-54401-0449",
        "spec_file": "spec-2682-54401-0449.fits",
        "label": "",
    },
    {  # F-star example
        "obs_id": "sdss_spectro_3298-54924-0214",
        "spec_file": "spec-3298-54924-0214.fits",
        "label": "",
    },
    {  # G-star example
        "obs_id": "sdss_spectro_1660-53230-0023",
        "spec_file": "spec-1660-53230-0023.fits",
        "label": "Sun-like Star. ",
    },
    {  # K-star example
        "obs_id": "sdss_spectro_3234-54885-0271",
        "spec_file": "spec-3234-54885-0271.fits",
        "label": "",
    },
    {  # M-star example
        "obs_id": "sdss_spectro_3298-54924-0159",
        "spec_file": "spec-3298-54924-0159.fits",
        "label": "",
    },
]

And here is a function to download the spectrum for each star using astroquery.MAST, just like we did earlier:

In [ ]:
def download_spec(star: dict[str, str]) -> str:
    """Helper function for downloading SDSS Legacy Spectra Data from MAST"""
    # Search for spectrum in astroquery
    obs_table = Observations.query_criteria(obs_id=star["obs_id"])
    # Retrieve product list
    products = Observations.get_product_list(obs_table)
    # Download spectrum
    manifest = Observations.download_products(
        products, flat=True, verbose=False, mrp_only=True
    )
    return manifest["Local Path"][0]

Now we're ready to plot all of our stars:

In [ ]:
fig = plt.figure(figsize=(12, 10))

# Define a custom rainbow colormap to use for the spectrum
rainbow_colormap = LinearSegmentedColormap.from_list(
    "",
    ["blueviolet"] + [mpl.cm.turbo(x) for x in np.linspace(0.07, 0.999, 10)],
)

for star_i, star in enumerate(star_list):
    print(f"Plotting star {star_i + 1}/{len(star_list)}...")
    # Download Spectrum File
    spectrum_file = download_spec(star)
    # Open spectrum file
    spectrum_file = fits.open(spectrum_file)
    classification = spectrum_file["SPECOBJ"].data["SUBCLASS"][0]

    # Initiate plot
    ax = plt.subplot(len(star_list), 1, star_i + 1)
    wavelength = 10 ** spectrum_file["COADD"].data["loglam"]
    observed_flux = spectrum_file["COADD"].data["FLUX"]
    model_flux = spectrum_file["COADD"].data["MODEL"]

    ax.plot(
        wavelength, observed_flux, c="w", label="Observed Spectrum", zorder=3
    )

    # Start with a black background
    ax.axvspan(np.min(wavelength), np.max(wavelength), color="k", zorder=0)

    plot_rainbow_background(ax, wavelength, observed_flux)

    # Label axes
    if star_i < 5:
        y = 0.90
        va = "top"
    else:
        y = 0.05
        va = "bottom"

    ax.text(
        0.99,
        y,
        f"{star['spec_file']}\n{star['label']}Classification: {classification}",
        fontsize=10,
        ha="right",
        va=va,
        color="w",
        transform=ax.transAxes,
    )

    # Add plot title
    if star_i == 0:
        ax.set_title("SEGUE Stellar Classification")

    # Set axes limits
    ax.set_xlim(np.min(wavelength), np.max(wavelength))
    ax.set_xscale("log")
    ax.set_ylim(-0.1, np.percentile(observed_flux, 99) * 1.1)
    ax.yaxis.set_visible(False)  # Hide y-axis labels

    if star_i == len(star_list) - 1:
        ax.set_xlabel(r"Wavelength ($\AA$)")
    else:
        ax.xaxis.set_visible(False)  # Hide x axis labels

# Adjust subplots so there is no white space between the spectra
plt.subplots_adjust(hspace=0)

# Display plot
plt.savefig("segue_spectra.png", bbox_inches="tight")
plt.show()

This shows what the spectra look like for different types of stars! We can see how the color varies from the hottest stars (top), which have the strongest purple, and the cooler stars (bottom), which have the strongest reds. A-type stars have the strongest Hydrogen lines, while M-type stars have giant absorption bands made by molecules. Now you know how to classify stars based on these example spectra!


### **Exercise 3:** Classify a random star

It's time to put everything we've learned together! Now that you know what different spectra of different types of stars look like, let's try classifying one by ourselves. To start, we will select a random star from SEGUE using the code in this cell:

In [ ]:
# Querying SDSS SEGUE data
spectro_data = Observations.query_criteria(
    provenance_name="SEGUE",
    target_classification="STAR",
    pagesize=100,
    page=1,
)

# Choose a random entry from the results:
i = np.random.choice(range(len(spectro_data)))
my_random_star = spectro_data[i]

# Display result
my_random_star

Now, it's time to practice what you've learned. In the following cell, write a bit of code that will download the spectrum for this star: 

In [ ]:
# Use astroquery.mast to download the spectrum for this star

And use it to make a plot:

In [ ]:
# Plot the spectrum

Compare this spectrum to the examples in the plot we made earlier in this notebook. What kind of star do you think this is? How can you check your answer?

In [ ]:
# After making a guess, see what type of star this is:

***

## Conclusions: Responsibility to Awe

>*We astronomers are nomads,*<br>
>*Merchants, circus people,*<br>
>*All the earth our tent.*<br>
>
>*We are industrious,*<br>
>*We breed enthusiasms,*<br>
>*Honour our responsibility to awe.*<br>
>
>*But the universe has moved a long way off.*<br>
>*Sometimes, I confess,*<br>
>*Starlight seems too sharp,*<br>
>
>*And like the moon*<br>
>*I bend my face to the ground,*<br>
>*To the small patch where each foot falls,*<br>
>
>*Before it falls,*<br>
>*And I forget to ask questions,*<br>
>*And only count things.*<br>
> <br>
> -Rebecca Elson, *We Astronomers*
> <br>
> <br>


### Exercise Solutions


In [ ]:
# EXERCISE 1 SOLUTION
coordinates = "308.81669, 76.546953"

# Search MAST for spectra data from SDSS of this star
spectro_data = Observations.query_criteria(
    coordinates=coordinates,
    radius=0,
    obs_collection="SDSS",
    dataproduct_type="spectrum",
)

# Retrieve the available products
products = Observations.get_product_list(spectro_data)

# Download the spectrum files
manifest = Observations.download_products(
    products,  # Specify a product list to download
    flat=True,  # Download with a "flat" structure in the current directory
    mrp_only=True,  # Limit to Minimum Recommended Products = the primary science file
)

In [ ]:
# EXERCISE 2 SOLUTION
# Open the file
spectrum_file = fits.open("hst_8143_stis_hd10700_e140m_o5cy_cspec.fits")

# Print some info about the file
spectrum_file.info()

# Read in the wavelength and flux values
wavelength = spectrum_file["SCI"].data["WAVELENGTH"][0]
observed_flux = spectrum_file["SCI"].data["FLUX"][0]

# Plot the spectrum using matplotlib here!
fig, ax = plt.subplots(figsize=(10, 4))

plt.plot(wavelength, observed_flux, c="gray", label="Observed Spectrum")

plt.grid()

# Label axes
plt.xlabel(r"Wavelength ($\AA$)")
plt.ylabel("Flux")
plt.legend()
plt.title(f"{spectrum_file.filename().strip('./')}")

# Set axes limits
plt.xlim(np.min(wavelength), np.max(wavelength))
plt.ylim(-1e-13, 1e-13)

# Show plot
plt.show()

In [ ]:
# EXERCISE 3 SOLUTION
# Querying SDSS SEGUE data
spectro_data = Observations.query_criteria(
    provenance_name="SEGUE",
    target_classification="STAR",
    pagesize=100,
    page=1,
)

# Choose a random entry from the results:
i = np.random.choice(range(len(spectro_data)))
my_random_star = spectro_data[i]

# Display result
my_random_star
# Use astroquery.mast to download the spectrum for this star
products = Observations.get_product_list(my_random_star)
manifest = Observations.download_products(
    products, flat=True, verbose=False, mrp_only=True
)

# Plot the spectrum
spectrum_file = fits.open(manifest["Local Path"][0])

fig, ax = plt.subplots(figsize=(10, 3))

wavelength = 10 ** spectrum_file["COADD"].data["loglam"]
observed_flux = spectrum_file["COADD"].data["FLUX"]
model_flux = spectrum_file["COADD"].data["MODEL"]

# Plot spectra
plt.plot(wavelength, observed_flux, c="gray", label="Observed Spectrum")
plt.plot(wavelength, model_flux, c="k", label="Model Spectrum")

# Label axes
ax.set_xlabel(r"Wavelength ($\AA$)")
ax.set_ylabel("Flux")
ax.set_title(f"{spectrum_file.filename().strip('./')}")
ax.grid()
plt.legend()

# Set axes limits
ax.set_xlim(np.min(wavelength), np.max(wavelength))
ax.set_ylim(0, 1.1 * np.max(model_flux))

# Show plot
plt.show()

# After making a guess, see what type of star this is:
classification = spectrum_file["SPECOBJ"].data["SUBCLASS"][0]
print(f"Classification: {classification}")

***

### Additional Resources

Additional resources are linked below:

- [SDSS Legacy Archive at MAST](https://archive.stsci.edu/missions-and-data/sdss)
- [SDSS Legacy Archive at MAST User Manual](https://outerspace.stsci.edu/display/SDSS/The+SDSS+Legacy+Archive+at+MAST)
- [SDSS Legacy Spectra User Manual](https://outerspace.stsci.edu/spaces/SDSS/pages/352354561/Legacy+Spectra+and+SEGUE)
- [astroquery.mast User Manual](https://astroquery.readthedocs.io/en/latest/mast/mast.html)
- [MAST API](https://mast.stsci.edu/api/v0/index.html)

### About this Notebook

This notebook was written for the 2026 MAST Summer webinar series.

**Author:** Julie Imig (jimig@stsci.edu)<br>
**Keywords:** Tutorial, SDSS, SEGUE, stars, spectra <br>
**First published:** June 2026 <br>
**Last updated:** June 2026 <br>

***
[Top of Page](#top)
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png" alt="Space Telescope Logo" width="200px"/> 